# 02_healthcare_fraud_mini_analysis

## Objective
Analyze healthcare claims data to identify anomaly-style patterns using SQL. This mini project focuses on suspicious claim timing, unusually high claim amounts, provider-wise comparisons, and member-level risk tagging.

## Project Scope
- detect short-gap repeated claims
- compare claims with provider averages
- compare claims with global average
- classify claims as above or below average
- create a final combined member-risk and provider-benchmark view

## Setup
The notebook below creates the sample tables and loads healthcare claims data before running the analysis queries.

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect(':memory:')
cur = con.cursor()

In [2]:
cur.execute("""
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    member_name TEXT,
    age INTEGER,
    gender TEXT
)
""")

cur.execute("""
CREATE TABLE providers (
    provider_id INTEGER PRIMARY KEY,
    provider_name TEXT,
    city TEXT
)
""")

cur.execute("""
CREATE TABLE claims (
    claim_id INTEGER PRIMARY KEY,
    member_id INTEGER,
    provider_id INTEGER,
    claim_date TEXT,
    claim_amount REAL,
    claim_status TEXT,
    FOREIGN KEY (member_id) REFERENCES members(member_id),
    FOREIGN KEY (provider_id) REFERENCES providers(provider_id)
)
""")
print("✅ Tables created")

✅ Tables created


In [3]:
members_data = [
    (1, 'John Doe', 35, 'M'),
    (2, 'Jane Smith', 29, 'F'),
    (3, 'James Brown', 42, 'M'),
    (4, 'Julia Roberts', 31, 'F'),
    (5, 'Jack Wilson', 27, 'M')
]

providers_data = [
    (1, 'UHC', 'New York'),
    (2, 'Aetna', 'Chicago'),
    (3, 'Cigna', 'Dallas'),
    (4, 'Humana', 'Boston')
]

claims_data = [
(201,1,1,'2025-10-01',12000,'Approved'),
(202,1,2,'2025-10-01',12000,'Pending'),
(203,1,3,'2025-10-05',18000,'Approved'),
(204,1,1,'2025-10-10',22000,'Denied'),
(205,1,2,'2025-10-15',22000,'Pending'),

(206,2,1,'2025-10-01',20000,'Approved'),
(207,2,1,'2025-10-15',15000,'Denied'),
(208,2,2,'2025-10-15',15000,'Pending'),
(209,2,3,'2025-11-01',10000,'Pending'),
(210,2,3,'2025-11-05',30000,'Approved'),

(211,3,2,'2025-10-12',25000,'Approved'),
(212,3,3,'2025-10-12',25000,'Denied'),
(213,3,1,'2025-10-20',25000,'Pending'),
(214,3,4,'2025-10-25',27000,'Approved'),
(215,3,2,'2025-11-02',25000,'Denied'),

(216,4,4,'2025-09-28',50000,'Pending'),
(217,4,3,'2025-10-10',45000,'Approved'),
(218,4,2,'2025-10-15',45000,'Denied'),
(219,4,1,'2025-10-20',30000,'Approved'),
(220,4,3,'2025-11-01',12000,'Pending'),

(221,5,2,'2025-10-05',8000,'Approved'),
(222,5,2,'2025-10-12',9000,'Pending'),
(223,5,3,'2025-10-18',7000,'Denied'),
(224,5,1,'2025-11-01',11000,'Approved'),
(225,5,1,'2025-11-05',18000,'Approved'),

(226,1,3,'2025-11-03',26000,'Approved'),
(227,2,2,'2025-11-02',14000,'Denied'),
(228,3,3,'2025-11-06',27000,'Approved'),
(229,4,4,'2025-11-07',32000,'Pending'),
(230,5,1,'2025-11-08',15000,'Approved')
]

cur.executemany("INSERT INTO members VALUES (?, ?, ?, ?);", members_data)
cur.executemany("INSERT INTO providers VALUES (?, ?, ?);", providers_data)
cur.executemany("INSERT INTO claims VALUES (?, ?, ?, ?, ?, ?);", claims_data)
con.commit()

print("✅ Sample data inserted")

✅ Sample data inserted


## Base Data Preview

In [4]:
print("Claims")
display(pd.read_sql_query("SELECT * FROM claims LIMIT 10;", con))

print("Members")
display(pd.read_sql_query("SELECT * FROM members;", con))

print("Providers")
display(pd.read_sql_query("SELECT * FROM providers;", con))

Claims


,claim_id,member_id,provider_id,claim_date,claim_amount,claim_status
0,201,1,1,2025-10-01,12000.0,Approved
1,202,1,2,2025-10-01,12000.0,Pending
2,203,1,3,2025-10-05,18000.0,Approved
3,204,1,1,2025-10-10,22000.0,Denied
4,205,1,2,2025-10-15,22000.0,Pending
5,206,2,1,2025-10-01,20000.0,Approved
6,207,2,1,2025-10-15,15000.0,Denied
7,208,2,2,2025-10-15,15000.0,Pending
8,209,2,3,2025-11-01,10000.0,Pending
9,210,2,3,2025-11-05,30000.0,Approved


Members


,member_id,member_name,age,gender
0,1,John Doe,35,M
1,2,Jane Smith,29,F
2,3,James Brown,42,M
3,4,Julia Roberts,31,F
4,5,Jack Wilson,27,M


Providers


,provider_id,provider_name,city
0,1,UHC,New York
1,2,Aetna,Chicago
2,3,Cigna,Dallas
3,4,Humana,Boston


## Query 1 - Short Gap Claim Flag

**Purpose:** Flag members who filed another claim within 2 days of the previous claim.

**Insight:** Very short gaps between claims may indicate unusual utilization patterns and are useful as an early fraud-style signal.

In [5]:
s1 = """WITH cte_prev AS(
                SELECT member_id,
                       claim_date,
                       LAG(claim_date) OVER(
                           PARTITION BY member_id
                           ORDER BY claim_date
                       ) AS previous_claim_date
                FROM claims
            ),
            cte_gap AS(
                SELECT member_id,
                       claim_date,
                       previous_claim_date,
                       CAST(julianday(claim_date) - julianday(previous_claim_date) AS INTEGER) AS gap_days
                FROM cte_prev
            )
            SELECT member_id,
                   claim_date,
                   previous_claim_date,
                   gap_days,
                   CASE
                       WHEN previous_claim_date IS NULL THEN 'First Claim'
                       WHEN gap_days <= 2 THEN 'Flag'
                       ELSE 'Normal'
                   END AS status_flag
            FROM cte_gap
            ORDER BY member_id, claim_date
      """
display(pd.read_sql_query(s1, con))

,member_id,claim_date,previous_claim_date,gap_days,status_flag
0,1,2025-10-01,None,NaN,First Claim
1,1,2025-10-01,2025-10-01,0.0,Flag
2,1,2025-10-05,2025-10-01,4.0,Normal
3,1,2025-10-10,2025-10-05,5.0,Normal
4,1,2025-10-15,2025-10-10,5.0,Normal
5,1,2025-11-03,2025-10-15,19.0,Normal
6,2,2025-10-01,None,NaN,First Claim
7,2,2025-10-15,2025-10-01,14.0,Normal
8,2,2025-10-15,2025-10-15,0.0,Flag
9,2,2025-11-01,2025-10-15,17.0,Normal


## Query 2 - Claim Amount Greater Than 2x Provider Average

**Purpose:** Compare each claim with its provider's average claim amount.

**Insight:** Claims that are more than 2 times the provider average can be reviewed as unusually high-value claims.

In [6]:
s2 = """WITH cte AS(
            SELECT provider_id,
                   claim_date,
                   claim_amount,
                   AVG(claim_amount) OVER(PARTITION BY provider_id) AS avg_provider_amount
            FROM claims
        )
        SELECT provider_id,
               claim_date,
               claim_amount,
               avg_provider_amount,
               CASE
                   WHEN claim_amount > 2 * avg_provider_amount THEN 'High'
                   ELSE 'Normal'
               END AS trend_status
        FROM cte
        ORDER BY provider_id, claim_date
     """
display(pd.read_sql_query(s2, con))

,provider_id,claim_date,claim_amount,avg_provider_amount,trend_status
0,1,2025-10-01,12000.0,18666.666667,Normal
1,1,2025-10-01,20000.0,18666.666667,Normal
2,1,2025-10-10,22000.0,18666.666667,Normal
3,1,2025-10-15,15000.0,18666.666667,Normal
4,1,2025-10-20,25000.0,18666.666667,Normal
5,1,2025-10-20,30000.0,18666.666667,Normal
6,1,2025-11-01,11000.0,18666.666667,Normal
7,1,2025-11-05,18000.0,18666.666667,Normal
8,1,2025-11-08,15000.0,18666.666667,Normal
9,2,2025-10-01,12000.0,19444.444444,Normal


## Query 3 - Global Average Claim Comparison

**Purpose:** Compare each claim with the overall average claim amount across the full dataset.

**Insight:** This creates a broad benchmark to identify which claims stand above the overall claims pattern.

In [7]:
s3 = """WITH cte AS(
            SELECT member_id,
                   claim_amount,
                   AVG(claim_amount) OVER() AS avg_claim_amount
            FROM claims
        )
        SELECT member_id,
               claim_amount,
               avg_claim_amount,
               CASE
                   WHEN claim_amount > avg_claim_amount THEN 'Above Avg'
                   ELSE 'Below Avg'
               END AS flag
        FROM cte
        ORDER BY member_id, claim_amount
     """
display(pd.read_sql_query(s3, con))

,member_id,claim_amount,avg_claim_amount,flag
0,1,12000.0,21733.333333,Below Avg
1,1,12000.0,21733.333333,Below Avg
2,1,18000.0,21733.333333,Below Avg
3,1,22000.0,21733.333333,Above Avg
4,1,22000.0,21733.333333,Above Avg
5,1,26000.0,21733.333333,Above Avg
6,2,10000.0,21733.333333,Below Avg
7,2,14000.0,21733.333333,Below Avg
8,2,15000.0,21733.333333,Below Avg
9,2,15000.0,21733.333333,Below Avg


## Query 4 - Provider-wise Average Comparison

**Purpose:** Compare each claim with the average claim amount of the same provider.

**Insight:** This is more business-relevant than a global comparison because it evaluates a claim against its own provider pattern.

In [8]:
s4 = """WITH cte AS(
            SELECT p.provider_name,
                   c.claim_amount,
                   AVG(c.claim_amount) OVER(PARTITION BY c.provider_id) AS avg_claim_amount
            FROM claims c
            JOIN providers p
              ON c.provider_id = p.provider_id
        )
        SELECT provider_name,
               claim_amount,
               avg_claim_amount,
               CASE
                   WHEN claim_amount > avg_claim_amount THEN 'Above Avg'
                   ELSE 'Below Avg'
               END AS flag
        FROM cte
        ORDER BY provider_name, claim_amount
     """
display(pd.read_sql_query(s4, con))

,provider_name,claim_amount,avg_claim_amount,flag
0,Aetna,8000.0,19444.444444,Below Avg
1,Aetna,9000.0,19444.444444,Below Avg
2,Aetna,12000.0,19444.444444,Below Avg
3,Aetna,14000.0,19444.444444,Below Avg
4,Aetna,15000.0,19444.444444,Below Avg
5,Aetna,22000.0,19444.444444,Above Avg
6,Aetna,25000.0,19444.444444,Above Avg
7,Aetna,25000.0,19444.444444,Above Avg
8,Aetna,45000.0,19444.444444,Above Avg
9,Cigna,7000.0,22222.222222,Below Avg


## Query 5 - Final Combined Member Risk and Provider Benchmark View

**Purpose:** Build one final review-ready query combining provider averages, member total claim amount, risk tagging, and claim flagging.

**Output Columns**
- member_name
- provider_name
- claim_amount
- avg_provider
- member_total_claim
- risk_tag
- flag

**Insight:** This final output combines provider-level and member-level signals in one place and can be used as a compact anomaly review table.

In [9]:
s5 = """WITH cte_provider_avg AS(
            SELECT m.member_id,
                   m.member_name,
                   p.provider_name,
                   c.claim_amount,
                   AVG(c.claim_amount) OVER(PARTITION BY p.provider_id) AS avg_provider
            FROM claims c
            JOIN providers p
              ON c.provider_id = p.provider_id
            JOIN members m
              ON c.member_id = m.member_id
        ),
        cte_member_total AS(
            SELECT member_name,
                   provider_name,
                   member_id,
                   avg_provider,
                   claim_amount,
                   SUM(claim_amount) OVER(PARTITION BY member_id) AS member_total_claim
            FROM cte_provider_avg
        )
        SELECT member_name,
               provider_name,
               claim_amount,
               avg_provider,
               member_total_claim,
               CASE
                   WHEN member_total_claim < 10000 THEN 'LOW'
                   WHEN member_total_claim BETWEEN 10000 AND 20000 THEN 'MEDIUM'
                   ELSE 'HIGH'
               END AS risk_tag,
               CASE
                   WHEN claim_amount > avg_provider THEN 'Above Avg'
                   ELSE 'Below Avg'
               END AS flag
        FROM cte_member_total
        ORDER BY member_name, provider_name, claim_amount
     """
display(pd.read_sql_query(s5, con))

,member_name,provider_name,claim_amount,avg_provider,member_total_claim,risk_tag,flag
0,Jack Wilson,Aetna,8000.0,19444.444444,68000.0,HIGH,Below Avg
1,Jack Wilson,Aetna,9000.0,19444.444444,68000.0,HIGH,Below Avg
2,Jack Wilson,Cigna,7000.0,22222.222222,68000.0,HIGH,Below Avg
3,Jack Wilson,UHC,11000.0,18666.666667,68000.0,HIGH,Below Avg
4,Jack Wilson,UHC,15000.0,18666.666667,68000.0,HIGH,Below Avg
5,Jack Wilson,UHC,18000.0,18666.666667,68000.0,HIGH,Below Avg
6,James Brown,Aetna,25000.0,19444.444444,154000.0,HIGH,Above Avg
7,James Brown,Aetna,25000.0,19444.444444,154000.0,HIGH,Above Avg
8,James Brown,Cigna,25000.0,22222.222222,154000.0,HIGH,Above Avg
9,James Brown,Cigna,27000.0,22222.222222,154000.0,HIGH,Above Avg


## Final Observations

- Short-gap repeat claims can be used as a timing-based review flag.
- Very high claims compared with provider averages are useful anomaly signals.
- Global average comparison provides a broad benchmark.
- Provider-wise comparison gives stronger business context.
- Member total claim amount supports simple risk segmentation.
- Combining multiple signals in one final query improves review efficiency.

## SQL Concepts Used
- JOIN
- CTE
- CASE WHEN
- LAG
- AVG() OVER()
- SUM() OVER()
- PARTITION BY
- `julianday()` for date gap analysis

## Conclusion
This mini project shows how SQL can be used for healthcare claim review, anomaly detection, and business-rule-driven fraud-style analysis.